# Runner dos experimentos (Colab gratuito)

Roda os experimentos completos e grava em `MyDrive/litm-results`. **É retomável:** se a sessão cair ou a cota de GPU acabar, reabra este notebook (no dia seguinte, se precisar) e rode tudo de novo. Ele continua de onde parou.

1. **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**
2. **Ambiente de execução → Executar tudo**
3. Autorize o acesso ao Google Drive quando pedir.

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
assert gpu, "Sem GPU: Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU"
print("GPU:", gpu)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/litm-results"
REPO = "/content/lost-in-the-middle-attention"

In [ ]:
!test -d {REPO} || git clone -q https://github.com/henrique-voni/lost-in-the-middle-attention {REPO}
!git -C {REPO} pull -q
!pip install -q -r {REPO}/requirements.txt

## Progresso até agora

In [ ]:
import os
import numpy as np
import pandas as pd

MODELS = ["phi4-mini", "smollm3-3b", "qwen3-4b"]

def status():
    for m in MODELS:
        a, c = f"{OUT}/attention_{m}.npz", f"{OUT}/accuracy_{m}.csv"
        att = int(np.load(a)["done"]) if os.path.exists(a) else 0
        acc = len(pd.read_csv(c)) if os.path.exists(c) else 0
        print(f"{m:11s} atenção {att:3d}/100 perguntas · acurácia {acc:4d}/1500 respostas")

status()

## Execução (atenção primeiro, que é mais barata; depois acurácia)

In [ ]:
%cd {REPO}
for exp, n in [("attention", 100), ("accuracy", 300)]:
    for m in MODELS:
        print(f"=== {m} · {exp} ===")
        !python scripts/run_experiments.py --model {m} --exp {exp} --n {n} --out {OUT}
status()